In [ ]:
"""
going to add a simple LLM synthesis step to the paper search system
because it provides an expensive operation worth optimizing.

then build semantic caching to avoid redundant API calls when similar 
questions are asked.

also implement conversation memory so system can handle follow-up queries
that reference earlier exchanges.

be able to use vector dbs for semantic caching and conversation memory.
these two methods reduce costs and improve multi-turn interactions in LLM apps.
"""

In [ ]:
# simple version of synthesis system to measure costs.
import os
import time
import hashlib

from numpy import dot
from numpy.linalg import norm

import cohere
import chromadb
import numpy as np
import pandas as pd
from dotenv import load_dotenv

In [2]:
load_dotenv()

True

In [3]:
### -----------------------------
### Global config and helpers
### -----------------------------
EMBED_MODEL = "embed-v4.0"
CHAT_MODEL = "command-a-03-2025"
CACHE_NAMESPACE = f"{CHAT_MODEL}|temp=default"

In [4]:
def embed_query(client: cohere.ClientV2, text: str) -> list[float]:
    """
    Generate an embedding for a query using Cohere ClientV2.
    Returns a list[float] suitable for ChromaDB.
    """
    resp = client.embed(
        model=EMBED_MODEL,
        input_type="search_query",
        texts=[text],
        embedding_types=["float"],
    )
    return resp.embeddings.float[0]

def extract_text(resp) -> str:
    """
    Extract generated text from Cohere chat responses.
    Works across Cohere response shapes.
    """
    return resp.text if hasattr(resp, "text") else resp.message.content[0].text

def reset_collection(client: chromadb.Client, name: str) -> None:
    """
    Delete and recreate a ChromaDB collection.
    Useful in notebooks to avoid duplicate ID errors on re-runs.
    """
    try:
        client.delete_collection(name)
    except Exception:
        pass

/Users/sbamwite/Desktop/rags/env/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


In [5]:
api_key = os.getenv("COHERE_API_KEY")
if not api_key:
    raise ValueError(
        "Missing COHERE_API_KEY. Add it to your .env file (COHERE_API_KEY=...) "
        "or set it as an environment variable before running this notebook."
    )


co = cohere.ClientV2(api_key)

In [8]:
chroma_client = chromadb.PersistentClient(path="./chroma_20260331_db")

In [7]:
papers_df = pd.read_csv("arxiv_papers_5k.csv")
embeddings = np.load("embeddings_cohere_5k.npy")

In [9]:
reset_collection(chroma_client, "arxiv_papers")

collection = chroma_client.get_or_create_collection(
    name="arxiv_papers",
    metadata={"hnsw:space": "cosine"},
)

In [10]:
batch_size = 5000
for i in range(0, len(papers_df), batch_size):
    batch_end = min(i + batch_size, len(papers_df))
    collection.add(
        ids=[str(idx) for idx in range(i, batch_end)],
        embeddings=embeddings[i:batch_end].tolist(),
        documents=papers_df["abstract"].iloc[i:batch_end].tolist(),
        metadatas=papers_df[["title", "category", "published"]]
        .iloc[i:batch_end]
        .to_dict("records"),
    )

In [11]:
### -----------------------------
### Baseline performance test
### -----------------------------
query = "What are attention mechanisms in transformers?"

### Time the embedding step
start = time.time()
query_embedding = embed_query(co, query)
embedding_time = (time.time() - start) * 1000

### Time the retrieval step
start = time.time()
results = collection.query(
    query_embeddings=[query_embedding],
    n_results=5,
    include=["documents", "metadatas", "distances"],
)
retrieval_time = (time.time() - start) * 1000

### Time the synthesis step
start = time.time()
papers_text = "\n\n".join([
    f"Paper {i+1}: {results['metadatas'][0][i]['title']}\n{results['documents'][0][i][:500]}..."
    for i in range(len(results["documents"][0]))
])

prompt = f"""Based on these research papers, answer the question: {query}

{papers_text}

Provide a clear, synthesized answer based on the papers above.
"""

resp = co.chat(
    model=CHAT_MODEL,
    messages=[{"role": "user", "content": prompt}],
)

_ = extract_text(resp)
synthesis_time = (time.time() - start) * 1000

total_time = embedding_time + retrieval_time + synthesis_time

print(f"Query: {query}")
print("\nTiming breakdown:")
print(f"  Embedding: {embedding_time:.1f}ms")
print(f"  Retrieval: {retrieval_time:.1f}ms")
print(f"  LLM Synthesis: {synthesis_time:.1f}ms")
print(f"  Total: {total_time:.1f}ms")
print(f"\nBottleneck: LLM synthesis is {synthesis_time/retrieval_time:.1f}x slower than retrieval")

Query: What are attention mechanisms in transformers?

Timing breakdown:
  Embedding: 987.4ms
  Retrieval: 52.8ms
  LLM Synthesis: 8632.3ms
  Total: 9672.5ms

Bottleneck: LLM synthesis is 163.5x slower than retrieval


In [ ]:
"""
Distance vs similarity:

ChromaDB returns distance, not similarity. When using cosine distance,
lower is better. A common pattern is to convert it to an approximate
similarity score using the formula below. Because cosine distance
ranges from 0 to 2, this converted “similarity” can be negative.
That simply means the queries are very dissimilar.

similarity ≈ 1 - distance
"""

In [13]:
class SemanticCache:
    def __init__(self, chroma_client, cohere_client):
        self.co = cohere_client
        self.cache_namespace = CACHE_NAMESPACE

        # Layer 1: Exact-match cache
        self.exact_cache = {}

        # Layer 2: Semantic cache
        self.semantic_cache = chroma_client.get_or_create_collection(
            name="query_cache",
            metadata={"hnsw:space": "cosine"},
        )

        self.cache_count = 0

    def _hash_query(self, query: str) -> str:
        base = f"{self.cache_namespace}:{query.lower().strip()}"
        return hashlib.md5(base.encode("utf-8")).hexdigest()

    def get(self, query: str, similarity_threshold: float = 0.90):
        # Layer 1: exact match
        query_hash = self._hash_query(query)
        if query_hash in self.exact_cache:
            return self.exact_cache[query_hash], "exact"

        # Layer 2: semantic match
        query_embedding = embed_query(self.co, query)

        results = self.semantic_cache.query(
            query_embeddings=[query_embedding],
            n_results=1,
            include=["documents", "metadatas", "distances"],
        )

        if not results.get("documents") or not results["documents"][0]:
            return None, None

        distance = results["distances"][0][0]
        similarity = 1 - distance

        if similarity >= similarity_threshold:
            cached_response = results["metadatas"][0][0]["response"]
            return cached_response, "semantic"

        return None, None

    def put(self, query: str, response: str) -> None:
        query_hash = self._hash_query(query)
        self.exact_cache[query_hash] = response

        query_embedding = embed_query(self.co, query)

        self.semantic_cache.add(
            ids=[f"cache_{self.cache_count}"],
            embeddings=[query_embedding],
            documents=[query],
            metadatas=[{"response": response}],
        )

        self.cache_count += 1

In [14]:
def answer_query_with_cache(
    query: str,
    cache: SemanticCache,
    collection,
    similarity_threshold: float = 0.90,
):
    start = time.time()

    cached_response, cache_type = cache.get(query, similarity_threshold=similarity_threshold)
    if cached_response is not None:
        elapsed = (time.time() - start) * 1000
        print(f"Cache hit ({cache_type}): {elapsed:.1f}ms")
        return cached_response, cache_type

    print("Cache miss - running full pipeline...")

    query_embedding = embed_query(cache.co, query)
    results = collection.query(
        query_embeddings=[query_embedding],
        n_results=5,
        include=["documents", "metadatas", "distances"],
    )

    papers_text = "\n\n".join([
        f"Paper {i+1}: {results['metadatas'][0][i]['title']}\n{results['documents'][0][i][:500]}..."
        for i in range(len(results["documents"][0]))
    ])

    prompt = f"""Based on these research papers, answer the question: {query} {papers_text} Provide a clear, synthesized answer based on the papers above."""

    resp = cache.co.chat(
        model=CHAT_MODEL,
        messages=[{"role": "user", "content": prompt}],
    )
    answer = extract_text(resp)

    cache.put(query, answer)

    elapsed = (time.time() - start) * 1000
    print(f"Full pipeline: {elapsed:.1f}ms")

    return answer, "miss"

In [15]:
try:
    chroma_client.delete_collection("query_cache")
except Exception:
    pass

cache = SemanticCache(chroma_client, co)

query1 = "What are attention mechanisms in transformers?"
answer1, _ = answer_query_with_cache(query1, cache, collection)
print(f"\nAnswer: {answer1[:200]}...")

Cache miss - running full pipeline...
Full pipeline: 11503.5ms

Answer: Attention mechanisms in transformers are a core component that enable the model to focus on relevant parts of the input sequence when processing each element. They compute a weighted sum of all elemen...


In [16]:
### Ask the exact same question
answer2, _ = answer_query_with_cache(query1, cache, collection)
print(f"\nSame answer returned: {answer1 == answer2}")

Cache hit (exact): 0.3ms

Same answer returned: True


In [17]:
### Ask a semantically similar question
query2 = "How do attention mechanisms work in transformer models?"
answer3, _ = answer_query_with_cache(query2, cache, collection, similarity_threshold=0.90)
print(f"\nGot cached response: {answer1 == answer3}")

Cache hit (semantic): 678.7ms

Got cached response: True


In [19]:
### Base query that's already cached
base_query = "What are attention mechanisms in transformers?"

### Test queries with different intents
test_queries = [
    # Natural rephrasing (SHOULD cache)
    "How do attention mechanisms work in transformer models?",
    "Explain attention in transformers",
    "What is the purpose of attention mechanisms?",
    "How does self-attention work?",

    # Different intent (should NOT cache)
    "How expensive are transformer models to train?",
    "What are transformer limitations?",
    "Why attention instead of RNNs?",
    "What datasets train transformers?",
]

### Embed queries using the same helper function we used earlier
base_embedding = embed_query(co, base_query)
test_embeddings = [embed_query(co, q) for q in test_queries]

print("Similarity scores to base query:")
print(f"Base: {base_query}\n")

for i, (query, emb) in enumerate(zip(test_queries, test_embeddings)):
    similarity = dot(base_embedding, emb) / (norm(base_embedding) * norm(emb))
    intent = "SHOULD cache" if i < 4 else "should NOT cache"
    print(f"{similarity:.4f} - {query} ({intent})")

Similarity scores to base query:
Base: What are attention mechanisms in transformers?

0.9449 - How do attention mechanisms work in transformer models? (SHOULD cache)
0.8416 - Explain attention in transformers (SHOULD cache)
0.7864 - What is the purpose of attention mechanisms? (SHOULD cache)
0.6004 - How does self-attention work? (SHOULD cache)
0.3596 - How expensive are transformer models to train? (should NOT cache)
0.3867 - What are transformer limitations? (should NOT cache)
0.5239 - Why attention instead of RNNs? (should NOT cache)
0.4553 - What datasets train transformers? (should NOT cache)


In [ ]:
"""
Choosing a Threshold;
Based on these similarity patterns, here are practical threshold recommendations:

0.95 (Conservative) Use when wrong answers are costly.
    This catches only very close paraphrasing. 
    some legitimate cache hits will be missed, but incorrect cached answers
    will never be return.
    Good for applications where accuracy matters more than cost savings.

0.90 (Balanced - Recommended) This is the sweet spot for most applications.
    It catches natural rephrasing while avoiding false positives.
    In the testing, this threshold distinguished between rephrased questions
    (0.84 to 0.94) and different questions (0.36 to 0.52) with zero false positives.

0.85 (Aggressive) Use when cost savings are critical.
    This catches more variations but approaches the danger zone where
    different questions might incorrectly hit the cache.
    Monitoring of cache hits carefully should be done at this threshold.
"""

In [20]:
### Simulate a realistic query workload
realistic_queries = [
    "What are attention mechanisms in transformers?",
    "How do attention mechanisms work in transformer models?",  # Variation
    "What are attention mechanisms in transformers?",  # Exact repeat
    "Explain the transformer architecture",
    "What is the transformer architecture?",  # Variation
    "How do transformers handle long sequences?",
    "What are the limitations of transformer models?",
    "Explain attention in transformers",  # Variation of query 1
    "How expensive are transformers to train?",
    "What datasets are used for training transformers?",
    "How do transformers compare to RNNs?",
    "What is self-attention?",
    "Explain the transformer architecture",  # Exact repeat
    "What are positional encodings in transformers?",
    "How do attention mechanisms work in transformer models?",  # Exact repeat
    "What are the computational costs of transformers?",
    "How do transformers handle variable length sequences?",
    "What are the key innovations in transformers?",
    "Explain attention in transformers",  # Exact repeat
    "What are attention mechanisms in transformers?",  # Exact repeat
    "How do transformers process text?",
    "What makes transformers effective for NLP?",
]

In [21]:
### Reset cache collection + cache object for a clean workload test
try:
    chroma_client.delete_collection("query_cache")
except Exception:
    pass

cache = SemanticCache(chroma_client, co)

### Track metrics
total_queries = len(realistic_queries)
exact_hits = 0
semantic_hits = 0
cache_misses = 0
total_time = 0

print("Processing realistic workload...\n")

for query in realistic_queries:
    start = time.time()

    # Try cache
    cached_response, cache_type = cache.get(query, similarity_threshold=0.90)

    if cached_response is not None:
        if cache_type == "exact":
            exact_hits += 1
        else:
            semantic_hits += 1

        elapsed = (time.time() - start) * 1000
        total_time += elapsed
        continue

    # Cache miss: run full pipeline
    cache_misses += 1

    query_embedding = embed_query(co, query)
    results = collection.query(
        query_embeddings=[query_embedding],
        n_results=5,
        include=["documents", "metadatas", "distances"],
    )

    papers_text = "\n\n".join([
        f"Paper {i+1}: {results['metadatas'][0][i]['title']}\n{results['documents'][0][i][:500]}..."
        for i in range(len(results["documents"][0]))
    ])

    prompt = f"""Based on these research papers, answer the question: {query}

{papers_text}

Provide a clear, synthesized answer based on the papers above.
"""

    resp = co.chat(
        model=CHAT_MODEL,
        messages=[{"role": "user", "content": prompt}],
    )

    answer = extract_text(resp)
    cache.put(query, answer)

    elapsed = (time.time() - start) * 1000
    total_time += elapsed

### Calculate metrics
hit_rate = ((exact_hits + semantic_hits) / total_queries) * 100
avg_time = total_time / total_queries

print(f"Workload Results (threshold=0.90):")
print(f"  Total queries: {total_queries}")
print(f"  Exact hits: {exact_hits} ({(exact_hits/total_queries)*100:.1f}%)")
print(f"  Semantic hits: {semantic_hits} ({(semantic_hits/total_queries)*100:.1f}%)")
print(f"  Cache misses: {cache_misses} ({(cache_misses/total_queries)*100:.1f}%)")
print(f"  Overall hit rate: {hit_rate:.1f}%")
print(f"\n  Average latency: {avg_time:.0f}ms per query")

### LLM calls avoided
llm_calls_avoided = total_queries - cache_misses

print(f"\n  LLM calls avoided: {llm_calls_avoided} ({(llm_calls_avoided/total_queries)*100:.1f}%)")

Processing realistic workload...

Workload Results (threshold=0.90):
  Total queries: 22
  Exact hits: 4 (18.2%)
  Semantic hits: 3 (13.6%)
  Cache misses: 15 (68.2%)
  Overall hit rate: 31.8%

  Average latency: 19277ms per query

  LLM calls avoided: 7 (31.8%)


In [ ]:
# NOTE
"""
If the answer depends on time, user identity, or session context,
do not reuse cached responses across requests.
"""

In [ ]:
"""
----------------------------------- The Multi-Turn Problem -----------------------------------
Turn 1: "What are attention mechanisms?"
→ System retrieves papers about attention mechanisms

Turn 2: "How do they compare to RNNs?"
→ Without memory: "they" = ??? (system has no idea)
→ With memory: "they" = attention mechanisms from Turn 1

Turn 3: "Show me efficient implementations"
→ Without memory: Implementations of WHAT?
→ With memory: Efficient attention mechanisms
"""

In [22]:
### First turn: Establish context
turn1_query = "What are attention mechanisms in transformers?"

### Retrieve papers to establish context (using our shared embedding helper)
turn1_embedding = embed_query(co, turn1_query)
turn1_results = collection.query(
    query_embeddings=[turn1_embedding],
    n_results=5,
    include=["metadatas"],
)

print("Turn 1 - Retrieved papers:")
for i, meta in enumerate(turn1_results["metadatas"][0]):
    print(f"{i+1}. {meta['title'][:80]}...")

### Second turn: Ambiguous follow-up
turn2_query = "Show me efficient implementations"

### WITHOUT MEMORY: search using the ambiguous query alone
print("\n\nTurn 2 WITHOUT MEMORY:")
print(f"Query: {turn2_query}")

turn2_embedding_no_memory = embed_query(co, turn2_query)
results_no_memory = collection.query(
    query_embeddings=[turn2_embedding_no_memory],
    n_results=5,
    include=["metadatas"],
)

print("Retrieved papers:")
for i, meta in enumerate(results_no_memory["metadatas"][0]):
    print(f"{i+1}. {meta['title'][:80]}...")

Turn 1 - Retrieved papers:
1. $π$-Attention: Periodic Sparse Transformers for Efficient Long-Context Modeling...
2. How Particle-System Random Batch Methods Enhance Graph Transformer: Memory Effic...
3. Multistability of Self-Attention Dynamics in Transformers...
4. A Unified Geometric Field Theory Framework for Transformers: From Manifold Embed...
5. Fractional neural attention for efficient multiscale sequence processing...


Turn 2 WITHOUT MEMORY:
Query: Show me efficient implementations
Retrieved papers:
1. Indexing Strings with Utilities...
2. MossNet: Mixture of State-Space Experts is a Multi-Head Attention...
3. Attention and Compression is all you need for Controllably Efficient Language Mo...
4. Hidden Sketch: A Space-Efficient Reversible Sketch for Tracking Frequent Items i...
5. Inferring the Most Similar Variable-length Subsequences between Multidimensional...


In [23]:
### WITH MEMORY: Include context from Turn 1
print("\n\nTurn 2 WITH MEMORY:")
print(f"Query: {turn2_query}")

### Create context-aware query by combining Turn 1 and Turn 2
memory_enhanced_query = f"{turn1_query} {turn2_query}"
print(f"Context-aware query: {memory_enhanced_query}")

turn2_embedding_with_memory = embed_query(co, memory_enhanced_query)

results_with_memory = collection.query(
    query_embeddings=[turn2_embedding_with_memory],
    n_results=5,
    include=["metadatas"],
)

print("\nRetrieved papers:")
for i, meta in enumerate(results_with_memory["metadatas"][0]):
    print(f"{i+1}. {meta['title'][:80]}...")

### Compare how many papers changed (using titles since IDs may not always be returned)
without_titles = {m["title"] for m in results_no_memory["metadatas"][0]}
with_titles = {m["title"] for m in results_with_memory["metadatas"][0]}
papers_changed = len(without_titles - with_titles)

print(f"\nPapers changed: {papers_changed} out of 5 ({(papers_changed/5)*100:.0f}%)")



Turn 2 WITH MEMORY:
Query: Show me efficient implementations
Context-aware query: What are attention mechanisms in transformers? Show me efficient implementations

Retrieved papers:
1. Attention and Compression is all you need for Controllably Efficient Language Mo...
2. $π$-Attention: Periodic Sparse Transformers for Efficient Long-Context Modeling...
3. FlashEVA: Accelerating LLM inference via Efficient Attention...
4. How Particle-System Random Batch Methods Enhance Graph Transformer: Memory Effic...
5. Making Every Head Count: Sparse Attention Without the Speed-Performance Trade-of...

Papers changed: 4 out of 5 (80%)


In [ ]:
# NOTE:
"""
Naively concatenating earlier turns can sometimes pull retrieval
toward whatever has been talked about previously. That is fine for learning,
but production systems usually limit memory to recent turns or store
summarized context instead.
"""

In [24]:
class ConversationMemory:
    def __init__(self, chroma_client, cohere_client):
        self.co = cohere_client
        self.turn_counter = 0

        # Create separate collection for memory
        self.memory_collection = chroma_client.get_or_create_collection(
            name="conversation_memory",
            metadata={"hnsw:space": "cosine"},
        )

    def add_turn(self, user_query: str, assistant_response: str) -> None:
        """Store a conversation turn in memory."""
        # Combine query and response for context
        # We use the first 200 chars of response to keep it manageable
        turn_text = f"User asked: {user_query}\nSystem answered: {assistant_response[:200]}..."

        # Embed and store (search_document is appropriate for stored content)
        embedding = self.co.embed(
            model=EMBED_MODEL,
            input_type="search_document",
            texts=[turn_text],
            embedding_types=["float"],
        ).embeddings.float[0]

        self.memory_collection.add(
            ids=[f"turn_{self.turn_counter}"],
            embeddings=[embedding],
            documents=[turn_text],
            metadatas=[{
                "turn": self.turn_counter,
                "user_query": user_query,
            }],
        )

        self.turn_counter += 1

    def get_relevant_context(self, current_query: str, n_results: int = 2) -> str:
        """Retrieve relevant past turns for the current query."""
        if self.turn_counter == 0:
            return ""  # No history yet

        # Embed current query using the shared query embedding helper
        query_embedding = embed_query(self.co, current_query)

        results = self.memory_collection.query(
            query_embeddings=[query_embedding],
            n_results=min(n_results, self.turn_counter),
            include=["documents", "metadatas", "distances"],
        )

        if not results.get("documents") or not results["documents"][0]:
            return ""

        # Format context from past turns
        context = "Previous conversation:\n"
        for doc in results["documents"][0]:
            context += f"{doc}\n\n"

        return context

### Test the memory system
memory = ConversationMemory(chroma_client, co)

print("Starting conversation with memory...\n")

### Turn 1
query1 = "What are attention mechanisms in transformers?"
response1 = "Attention mechanisms allow transformers to weigh different parts of the input..."
memory.add_turn(query1, response1)
print(f"Turn 1: {query1}")
print(f"Response: {response1[:50]}...\n")

### Turn 2
query2 = "How do they compare to RNNs?"
context = memory.get_relevant_context(query2)

print(f"Turn 2: {query2}")
print(f"Retrieved context:\n{context}")


Starting conversation with memory...

Turn 1: What are attention mechanisms in transformers?
Response: Attention mechanisms allow transformers to weigh d...

Turn 2: How do they compare to RNNs?
Retrieved context:
Previous conversation:
User asked: What are attention mechanisms in transformers?
System answered: Attention mechanisms allow transformers to weigh different parts of the input......




In [ ]:
"""
Production systems might experiment with alternatives
like storing just the user query, storing full responses,
or storing LLM-generated summaries of turns. 
"""

In [ ]:
"""
Key Takeaways:

- LLM synthesis is the bottleneck: Embedding and vector retrieval are fast,
  but synthesis takes seconds, making it the most valuable target for optimization.

- Two-tier caching works well in practice: Exact match caching is nearly free and catches repeated queries.
  Semantic caching is slower than exact match but still far cheaper than calling the LLM again.

- Semantic thresholds are a tradeoff: High thresholds reduce wrong cache hits but miss legitimate rephrases.
  Low thresholds increase hit rates but risk incorrect reuse.

- Most savings often come from exact repeats: In exploratory research workloads,
  many cache hits come from users repeating the same question, not just paraphrasing it.

- Guardrails matter for correctness: Queries that depend on time, user identity,
  or session context should bypass caching to avoid incorrect responses.

- Conversation memory improves retrieval: Follow-up queries like
  “Show me efficient implementations” become meaningfully searchable when you include context from earlier turns.

- Start simple and measure: Use basic patterns first, then refine thresholds, guardrails,
  and memory strategies based on real usage metrics.
"""